# 05. 지연 측정과 pgvector 재현성

**이 노트북만 로컬 Mac에서 실행한다.** Colab이 Drive `runs/`에 남긴 산출물을 동기화 폴더에서 읽는다. 로컬에서 잴 수 있는 구간만 실측한다. torch 스레드 2개(운영 VM 코어 수), pgvector는 Docker `--cpus=2` 컨테이너다. warm-up을 뺀 500회 p50/p95를 기록한다. Mac의 절대값이 운영 VM과 같다고 주장하지 않고 **비교군 사이의 상대 비교**에 쓴다.

| 구간 | 방법 |
|---|---|
| exact 조회 | 운영 `SELECT_EXACT_SQL` (CacheService.java에서 직접 읽음) |
| BGE 임베딩 | 한 문장씩, CPU |
| 벡터 검색 | 운영 `SELECT_SQL`, HNSW, 캐시 크기 500/2,000/4,000 |
| 리랭커 | 후보 5개 배치 |
| LLM 생성, 서비스 간 왕복 | 측정 불가. 생성은 xRouteBench 기록 시간만, 왕복은 8월 운영 관측 120ms를 **가정값**으로 06에서 사용 |

추가로 HNSW 상위 5개가 정확 검색과 같은 비율(recall), 그리고 D1 trace 하나를 **실제 pgvector + 운영 SQL**로 재생해 메모리 재생과 판정이 같은지 확인한다.

In [ ]:
import hashlib
import json
import os
import subprocess
import sys
import zipfile
from pathlib import Path

RUN_ID = os.environ.get("CACHE_EXP_RUN_ID", "run001")  # 새 계획·재시작이면 새 ID
IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    LLM_ROUTER = Path("/content/drive/MyDrive/LLM_Router")
    KIT = LLM_ROUTER / "cache-experiment"
    zip_path = KIT / "source/cache-source.zip"
    expected = (KIT / "source/cache-source.sha256").read_text().split()[0]
    assert hashlib.sha256(zip_path.read_bytes()).hexdigest() == expected, "source ZIP SHA mismatch"
    REPO = Path("/content/cache-source")
    with zipfile.ZipFile(zip_path) as z:
        for name in z.namelist():
            assert (REPO / name).resolve().is_relative_to(REPO.resolve()), name
        z.extractall(REPO)
    for name, digest in json.loads((REPO / "source-bundle-manifest.json").read_text()).items():
        assert hashlib.sha256((REPO / name).read_bytes()).hexdigest() == digest, name
    HERE = REPO / "ml/notebooks/cache_experiment"
    DATA_ROOT, OUTPUT_ROOT = KIT / "data-root", KIT / "runs"
    ROUTER_DIR = LLM_ROUTER / "transfer-experiments/inputs/experiment-A/models/r0"
    os.environ.setdefault("CACHE_EXP_DRIVE_LOGGER", str(LLM_ROUTER / "retraining-effect/logger/logger-log.parquet"))
    assert sys.version_info[:2] in {(3, 11), (3, 12)}, "런타임 버전 2026.07(Python 3.12)을 선택하세요"
    if not (REPO / ".installed").exists():
        subprocess.run([sys.executable, "-m", "pip", "uninstall", "-y", "torchvision", "torchaudio"], check=True)
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", "torch==2.5.1",
                        "--index-url", "https://download.pytorch.org/whl/cu124"], check=True)
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", str(HERE / "requirements-colab.txt")], check=True)
        (REPO / ".installed").write_text("ok")
        # Colab already imported its own numpy; mixing it with the pinned one breaks
        # transformers. Restart the runtime (files in /content survive) and run all again.
        print("설치 완료. 런타임을 재시작합니다. 재시작되면 '모두 실행'을 다시 누르세요.")
        os.kill(os.getpid(), 9)
    import numpy
    pinned = next(l.split("==")[1].strip() for l in (HERE / "requirements-colab.txt").read_text().splitlines()
                  if l.startswith("numpy=="))
    assert numpy.__version__ == pinned, (
        f"numpy {numpy.__version__} != {pinned}: 런타임 → 세션 다시 시작 후 '모두 실행'을 다시 누르세요")
else:
    # 로컬: 저장소 루트의 .venv-cache 커널. 입력은 ml/data·ml/artifacts가 있는 체크아웃,
    # 산출물은 Drive 동기화 폴더의 kit runs/ 가 있으면 그곳(Colab과 같은 run을 이어 쓴다).
    HERE = Path.cwd()
    DATA_ROOT = Path(os.environ.get("CACHE_EXP_DATA_ROOT", HERE.parents[2]))
    synced = sorted(Path.home().glob("Library/CloudStorage/GoogleDrive-*/*/LLM_Router/cache-experiment/runs"))
    OUTPUT_ROOT = os.environ.get("CACHE_EXP_OUTPUT_ROOT") or (synced[0] if synced else None)
    ROUTER_DIR = None  # config router.package_path
    found = sorted(Path.home().glob("Library/CloudStorage/GoogleDrive-*/*/LLM_Router/retraining-effect/logger/logger-log.parquet"))
    if found:
        os.environ.setdefault("CACHE_EXP_DRIVE_LOGGER", str(found[0]))
CONFIG = Path(os.environ.get("CACHE_EXP_CONFIG", HERE / "experiment_config.json"))
sys.path.insert(0, str(HERE))

import pandas as pd
import cache_experiment as ce
import models
import pipeline

pd.set_option("display.max_columns", 60)
pd.set_option("display.width", 200)
run = pipeline.Run(DATA_ROOT, RUN_ID, CONFIG, OUTPUT_ROOT)
print("run directory:", run.root)

In [ ]:
runtime = models.runtime_versions(run.config)
assert not IN_COLAB, "05는 로컬에서 실행한다: 운영 VM처럼 CPU 2스레드로 재고 Docker pgvector가 필요하다"
CPU = run.config["latency"]["device"]
embedder = models.Embedder(run.config, device=CPU)
reranker = models.Reranker(run.config, device=CPU)
router = models.Router(run.config, run.data_root, device=CPU)

In [ ]:
import latency
latency.stop_postgres()
dsn = latency.start_postgres(run.config)
latency.operational_sql()

라우터(운영 R0 패키지)는 미스 경로의 지연을 재려고 적재한다. 표본 100건을 CPU로 다시 분류해 00(Colab GPU)의 판정과 같은지 함께 기록한다.

In [ ]:
pipeline.router_spot_check(run, router)

In [ ]:
model_latency = pipeline.latency_models(run, embedder, reranker, router)
model_latency.round(2)

In [ ]:
pg_latency, recall = pipeline.latency_pgvector(run, dsn)
display(pg_latency.round(3))
recall

In [ ]:
PARITY_TRACE = ce.trace_id(0.3, run.config["d1"]["trace_seeds"][0])
pipeline.pgvector_decision_parity(run, dsn, reranker, PARITY_TRACE)

In [ ]:
latency.stop_postgres()